## Firm Size and Job-Based Social Protections: 10 Graphs

#### Jobs, Workers, and Development (Fall 2026) | Georgetown University | Emilia Antunez

**Research question:** Does the size of a firm affect how many social protections workers receive?

**Data:** `firmsize_clean_data.csv`, the output of `Firmsize_CPS_data_prep_1996_2026.ipynb`
(CPS ASEC 1996–2026, private wage and salary workers aged 16–64 who worked last year).

**Firm size, 4 groups:** under 10, 10–99, 100–499 and 500+ workers.

**Run the prep notebook first** so `firmsize_clean_data.csv` in My Drive has all four firm-size groups and every year.

**Weights:** `asecwt` for employer health insurance, pension and every covariate; `earnwt` for union (earner study only).
Before each graph, the notebook prints the weighted table the graph shows (1 decimal).

**Output:** 10 PNG graphs + a zip, saved to `My Drive/Jobs, Workers, and Development/Project/graphs/`,
a copy of this notebook saved to the Project folder, and the zip downloaded to your computer.

Run everything with **Runtime → Run all**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

PROJECT = '/content/drive/My Drive/Jobs, Workers, and Development/Project'

# firmsize_clean_data.csv is saved by the prep notebook to My Drive; also look in the Project folder
candidates = ['/content/drive/My Drive/firmsize_clean_data.csv',
              f'{PROJECT}/firmsize_clean_data.csv'] + \
             glob.glob('/content/drive/My Drive/Jobs, Workers, and Development/**/firmsize_clean_data.csv', recursive=True)
path = next(p for p in candidates if os.path.exists(p))

df = pd.read_csv(path)
df.columns = [c.lower() for c in df.columns]   # this dataset uses lowercase names
print('Loaded', path)
print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')
print(df.columns.tolist())

## Setup: style, labels, and weighted helpers

In [ ]:
# ---------- Style ----------
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948')
COLORS = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED]
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
BLUES = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b']

plt.rcParams.update({
    'figure.figsize': (10, 5.5), 'figure.dpi': 110, 'savefig.dpi': 200,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': GRID, 'axes.labelcolor': INK2, 'axes.titlesize': 15,
    'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'axes.titlecolor': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'xtick.color': INK2, 'ytick.color': INK2, 'font.size': 11,
    'lines.linewidth': 2.2, 'legend.frameon': False,
})
pct = mtick.PercentFormatter(100, decimals=0)

def pct_axis(ax, axis='y'):
    a = ax.yaxis if axis == 'y' else ax.xaxis
    a.set_major_locator(mtick.MaxNLocator(steps=[1, 2, 5, 10]))   # round tick steps (5%, 10%, ...)
    a.set_major_formatter(pct)

def year_ticks(ax, years):
    ax.set_xticks([y for y in range(1995, int(years.max()) + 1, 5) if y >= years.min() - 1])

OUT = '/content/graphs'
os.makedirs(OUT, exist_ok=True)
saved = []

# ---------- Labels ----------
# Builds readable groups from whatever version of firmsize_clean_data.csv you have:
# a *_label column if it exists, otherwise the number codes, otherwise the 0/1 dummies.
OUTCOMES = {'emp_hins': 'Employer health insurance', 'pension': 'Pension plan', 'union': 'Union member or covered'}
OUT_COLORS = {'emp_hins': BLUE, 'pension': ORANGE, 'union': AQUA}

def build_groups(new, label_col, code_col, codes, dummies):
    """Make column `new` with labels in a fixed order. Returns the order used ([] if impossible)."""
    order = list(codes.values())
    if code_col in df.columns:
        df[new] = df[code_col].map(codes)
    elif label_col in df.columns:
        df[new] = df[label_col]
        order = [o for o in order if o in set(df[new].dropna())] or sorted(df[new].dropna().unique())
    elif dummies and all(d in df.columns for d in dummies):
        df[new] = None
        for d, lab in zip(dummies, order):
            df.loc[df[d] == 1, new] = lab
    else:
        return []
    order = [o for o in order if o in set(df[new].dropna())]
    df[new] = pd.Categorical(df[new], order, ordered=True)
    print(f'{new}: built from', code_col if code_col in df.columns else label_col if label_col in df.columns else 'dummies', '→', order)
    return order

# Firm size: 4 groups (under 10 / 10-99 / 100-499 / 500+), same as the prep notebook
if 'firmsize4_label' in df.columns:
    df['firmsize4_label'] = df['firmsize4_label'].replace({'11-100': '10-99'})   # older label for the same group
FIRM_ORDER = build_groups('firm', 'firmsize4_label', 'firmsize4',
                          {1: 'Under 10', 2: '10-99', 3: '100-499', 4: '500+'},
                          ['micro', 'small', 'medium', 'large'])
if len(FIRM_ORDER) != 4:
    raise ValueError(
        'firmsize_clean_data.csv does not have the 4 firm-size groups (firmsize4). '
        'It is an older version of the file. Run Firmsize_CPS_data_prep_1996_2026.ipynb again '
        '(it saves the new file to My Drive), then run this notebook again. '
        f'Columns found: {df.columns.tolist()}')
FIRM_COLORS = dict(zip(FIRM_ORDER, [ORANGE, YELLOW, AQUA, BLUE]))
SMALLEST, LARGEST = FIRM_ORDER[0], FIRM_ORDER[-1]

EDUC_ORDER = build_groups('educ', 'educ4_label', 'educ4',
                          {1: 'Less than HS', 2: 'High school', 3: 'Some college', 4: 'BA or more'},
                          ['lths', 'hs', 'somecol', 'baplus'])
RACE_ORDER = build_groups('race', 'race5_label', 'race5',
                          {1: 'White', 2: 'Black', 3: 'Asian', 4: 'Hispanic', 5: 'Other'},
                          ['white', 'black', 'asian', 'hispanic', 'other_race'])
IND_ORDER = build_groups('ind', 'industry_label', 'industry',
                         {1: 'Agriculture', 2: 'Mining', 3: 'Construction', 4: 'Manufacturing',
                          5: 'Transport/communications/utilities', 6: 'Wholesale/retail trade',
                          7: 'Finance/insurance/real estate', 8: 'Business and repair services',
                          9: 'Personal services', 10: 'Entertainment/recreation',
                          11: 'Professional services', 12: 'Public administration'}, [])
if 'age' in df.columns:
    df['age_group'] = pd.cut(df['age'], [15, 24, 34, 44, 54, 64],
                             labels=['16-24', '25-34', '35-44', '45-54', '55-64'])

def has(*cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f'⚠️  Skipping this graph: missing column(s) {missing}')
    return not missing

# ---------- Weighted share (%) ----------
def weight_for(var):
    return 'earnwt' if var == 'union' else 'asecwt'   # union is earner-study only

def wshare(var, by, data=None):
    """Weighted % of workers with var == 1, by group(s). Uses earnwt for union, asecwt otherwise."""
    d = df if data is None else data
    w = weight_for(var)
    d = d[d[var].notna() & (d[w] > 0)]
    num = (d[var] * d[w]).groupby([d[b] for b in np.atleast_1d(by)], observed=True).sum()
    den = d[w].groupby([d[b] for b in np.atleast_1d(by)], observed=True).sum()
    return 100 * num / den

def show_table(t, title, fmt='{:.1f}'):
    print(title)
    print(t.to_string(float_format=fmt.format))
    print()

def subtitle(ax, text):
    ax.text(0, 1.02, text, transform=ax.transAxes, color=INK2, fontsize=10.5, va='bottom')

def label_end(ax, x, y, text):
    ax.annotate(text, (x, y), xytext=(6, 0), textcoords='offset points', va='center', color=INK, fontsize=9.5)

YEARS = (int(df['year'].min()), int(df['year'].max()))
SOURCE = (f'Source: IPUMS-CPS ASEC {YEARS[0]}-{YEARS[1]}, private wage and salary workers aged 16-64. '
          'Weighted: asecwt (health insurance, pension), earnwt (union).')

def finish(fig, name):
    fig.text(0.01, 0.005, SOURCE, color=INK2, fontsize=8, ha='left', va='bottom')
    fig.tight_layout(rect=(0, 0.035, 1, 1))
    file = f'{OUT}/{name}.png'
    fig.savefig(file, bbox_inches='tight')
    saved.append(file)
    plt.show()
    print('Saved', file)

from matplotlib.colors import LinearSegmentedColormap
CMAP = LinearSegmentedColormap.from_list('blues', BLUES)

def heatmap(t, title, sub, xlabel, ylabel, file, cbar_label, figsize=(10, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(t.values, cmap=CMAP, aspect='auto', vmin=0, vmax=100)
    ax.set_xticks(range(t.shape[1]), t.columns.astype(str))
    ax.set_yticks(range(t.shape[0]), t.index.astype(str))
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=10,
                        color='white' if v > 55 else INK)
    ax.grid(False)
    for sp in ax.spines.values(): sp.set_visible(False)
    cb = fig.colorbar(im, ax=ax, format=pct, shrink=0.85)
    cb.outline.set_visible(False)
    cb.set_label(cbar_label)
    ax.set_title(title, pad=24)
    subtitle(ax, sub)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    finish(fig, file)

print('Years in the data:', YEARS)
if YEARS != (1996, 2026):
    print(f'⚠️  The data covers {YEARS[0]}-{YEARS[1]}, not 1996-2026. Re-run Firmsize_CPS_data_prep_1996_2026.ipynb '
          'to get every year IPUMS has released (it prints which years it could not get).')
print('Ready ✔')

## 1. The headline: protections by firm size in 1996, 2006, 2016 and 2026
Share of workers with each protection, by firm size, in four years ten years apart.
Each panel is one protection; the bars go from light (1996) to dark (2026).

In [ ]:
WANTED = [1996, 2006, 2016, 2026]
SNAP_YEARS = [y for y in WANTED if y in set(df['year'])]
if len(SNAP_YEARS) < len(WANTED):
    print(f'⚠️  Not in the data, left out: {sorted(set(WANTED) - set(SNAP_YEARS))}')
YEAR_COLORS = dict(zip(SNAP_YEARS, ['#86b6ef', '#3987e5', '#1c5cab', '#0d366b']))   # light → dark = older → newer

snap = df[df['year'].isin(SNAP_YEARS)]
tables = {}
for v, name in OUTCOMES.items():
    t = wshare(v, ['year', 'firm'], data=snap).unstack()[FIRM_ORDER]
    t.index.name, t.columns.name = 'Year', 'Firm size (workers)'
    tables[v] = t
    show_table(t, f'{name} (% of workers), by year and firm size, weighted with {weight_for(v)}')

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
x = np.arange(len(FIRM_ORDER))
width = 0.8 / len(SNAP_YEARS)
for ax, (v, name) in zip(axes, OUTCOMES.items()):
    t = tables[v]
    for i, yr in enumerate(SNAP_YEARS):
        bars = ax.bar(x + (i - (len(SNAP_YEARS) - 1) / 2) * width, t.loc[yr], width=width - 0.02,
                      color=YEAR_COLORS[yr], label=str(yr))
        ax.bar_label(bars, fmt='%.0f', padding=2, fontsize=7.5, color=INK2)
    ax.set_xticks(x, FIRM_ORDER)
    ax.set_ylim(0, t.max().max() * 1.2)
    pct_axis(ax)
    ax.grid(axis='x', visible=False)
    ax.set_title(name, fontsize=13, pad=8)
    ax.set_xlabel('Firm size (number of workers)')
axes[0].set_ylabel('Share of workers (%)')
axes[0].legend(title='Survey year', loc='upper left', ncol=len(SNAP_YEARS), fontsize=9)
fig.suptitle('Protections by firm size, ' + ', '.join(map(str, SNAP_YEARS)),
             x=0.01, ha='left', fontsize=16, fontweight='bold', color=INK)
finish(fig, '01_protections_by_firm_size_1996_2026')

## 2. Employer health insurance over time, by firm size
Dotted lines mark survey changes: 2014 (questionnaire redesign) and 2019 (new universe and imputation). The 2001 verification question is already adjusted in the prep notebook.

In [ ]:
def trend_by_firm(var, name, file, title):
    t = wshare(var, ['year', 'firm']).unstack()[FIRM_ORDER]
    t.index.name = 'Year'
    show_table(t, f'{OUTCOMES[var]} (% of workers), by year and firm size, weighted with {weight_for(var)}')
    fig, ax = plt.subplots()
    for g in FIRM_ORDER:
        ax.plot(t.index, t[g], color=FIRM_COLORS[g], label=g, marker='o', markersize=3.5)
        label_end(ax, t.index[-1], t[g].iloc[-1], g)
    for yr in (2014, 2019):
        if t.index.min() < yr <= t.index.max():
            ax.axvline(yr - 0.5, color=INK2, linestyle=':', linewidth=1)
            ax.text(yr - 0.4, 2, f'{yr} survey change', color=INK2, fontsize=8.5, rotation=90, va='bottom')
    ax.set_ylim(0, max(100 if t.max().max() > 60 else t.max().max() * 1.3, 1))
    pct_axis(ax)
    ax.set_xlim(right=t.index[-1] + 3)
    year_ticks(ax, t.index)
    ax.legend(title='Firm size (workers)', loc='upper left', ncol=4, fontsize=9.5)
    ax.set_title(title, pad=24)
    subtitle(ax, f'Share of workers, by firm size')
    ax.set_xlabel('Survey year')
    ax.set_ylabel('Share of workers (%)')
    finish(fig, file)

trend_by_firm('emp_hins', OUTCOMES['emp_hins'], '02_health_insurance_trend_by_firm_size',
              'Employer health insurance by firm size, over time')

## 3. Pension plan coverage over time, by firm size

In [ ]:
trend_by_firm('pension', OUTCOMES['pension'], '03_pension_trend_by_firm_size',
              'Pension plan coverage by firm size, over time')

## 4. Union coverage over time, by firm size
Union is asked only in the earner study, so this graph uses `earnwt`.

In [ ]:
trend_by_firm('union', OUTCOMES['union'], '04_union_trend_by_firm_size',
              'Union coverage by firm size, over time')

## 5. Is the firm-size gap shrinking or growing?
The gap = share in the largest firms (500+) minus share in the smallest firm-size group, in percentage points.

In [ ]:
gap = pd.DataFrame({
    OUTCOMES[v]: (lambda s: s[LARGEST] - s[SMALLEST])(wshare(v, ['year', 'firm']).unstack())
    for v in OUTCOMES})
gap.index.name = 'Year'
show_table(gap, f'Gap: {LARGEST} firms minus {SMALLEST} firms (percentage points), weighted')

fig, ax = plt.subplots()
for v, name in OUTCOMES.items():
    ax.plot(gap.index, gap[name], color=OUT_COLORS[v], label=name, marker='o', markersize=3.5)
    label_end(ax, gap.index[-1], gap[name].iloc[-1], name)
ax.set_ylim(0, gap.max().max() * 1.25)
ax.set_xlim(right=gap.index[-1] + 6)
year_ticks(ax, gap.index)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f'{y:.0f} pp'))
ax.legend(loc='upper left', ncol=3, fontsize=9.5)
ax.set_title('The large-vs-small firm gap in protections', pad=24)
subtitle(ax, f'Share in firms with {LARGEST} workers minus share in firms with {SMALLEST.lower()} workers')
ax.set_xlabel('Survey year')
ax.set_ylabel('Gap (percentage points)')
finish(fig, '05_firm_size_gap_over_time')

## 6. Where do private-sector workers work? Share of workers by firm size over time

In [ ]:
t = df.groupby(['year', 'firm'], observed=True)['asecwt'].sum().unstack()[FIRM_ORDER]
t = 100 * t.div(t.sum(axis=1), axis=0)
t.index.name = 'Year'
show_table(t, 'Share of workers (%) by firm size, weighted with asecwt (rows add to 100)')

fig, ax = plt.subplots()
ax.stackplot(t.index, *[t[g] for g in FIRM_ORDER], colors=[FIRM_COLORS[g] for g in FIRM_ORDER],
             labels=FIRM_ORDER, edgecolor='white', linewidth=1)
bottom = 0
for g in FIRM_ORDER:   # label each band at the right edge
    mid = bottom + t[g].iloc[-1] / 2
    label_end(ax, t.index[-1], mid, f'{g}: {t[g].iloc[-1]:.0f}%')
    bottom += t[g].iloc[-1]
ax.set_ylim(0, 100)
ax.set_xlim(t.index[0], t.index[-1] + 4)
year_ticks(ax, t.index)
pct_axis(ax)
ax.legend(title='Firm size (workers)', loc='upper center', bbox_to_anchor=(0.45, -0.12), ncol=4)
ax.set_title('Share of private-sector workers by firm size', pad=24)
subtitle(ax, 'Each year adds to 100%')
ax.set_xlabel('Survey year')
ax.set_ylabel('Share of workers (%)')
finish(fig, '06_worker_share_by_firm_size')

## 7. Does firm size matter even for workers with the same education? (heatmap)
Employer health insurance by education and firm size. Darker = more coverage.

In [ ]:
if EDUC_ORDER:
    t = wshare('emp_hins', ['educ', 'firm']).unstack()[FIRM_ORDER]
    t.index.name, t.columns.name = 'Education', 'Firm size (workers)'
    show_table(t, 'Employer health insurance (% of workers), by education and firm size, weighted with asecwt')
    heatmap(t, 'Health insurance by education and firm size',
            'Share of workers with employer health insurance (%), all years',
            'Firm size (number of workers)', 'Education', '07_heatmap_insurance_education_firm_size',
            'Share of workers (%)')
else:
    print('⚠️  Skipping this graph: no education column in the data')

## 8. Is it firm size, or the industry? (heatmap)
Employer health insurance by industry and firm size. If the color still gets darker left → right inside each industry, firm size matters beyond industry.

In [ ]:
if IND_ORDER:
    t = wshare('emp_hins', ['ind', 'firm']).unstack()[FIRM_ORDER]
    t = t.loc[t.mean(axis=1).sort_values(ascending=False).index]
    t.index.name, t.columns.name = 'Industry', 'Firm size (workers)'
    show_table(t, 'Employer health insurance (% of workers), by industry and firm size, weighted with asecwt')
    heatmap(t, 'Health insurance by industry and firm size',
            'Share of workers with employer health insurance (%), all years. Industries sorted by average coverage',
            'Firm size (number of workers)', '', '08_heatmap_insurance_industry_firm_size',
            'Share of workers (%)', figsize=(10, 7.5))
else:
    print('⚠️  Skipping this graph: no industry column in the data')

## 9. Pension coverage by race/ethnicity and firm size
Does the firm-size gap look the same for every group?

In [ ]:
if RACE_ORDER:
    t = wshare('pension', ['race', 'firm']).unstack()[FIRM_ORDER]
    t.index.name, t.columns.name = 'Race / ethnicity', 'Firm size (workers)'
    show_table(t, 'Pension plan (% of workers), by race/ethnicity and firm size, weighted with asecwt')

    x = np.arange(len(t))
    k = len(FIRM_ORDER)
    width = 0.8 / k
    fig, ax = plt.subplots()
    for i, g in enumerate(FIRM_ORDER):
        bars = ax.bar(x + (i - (k - 1) / 2) * width, t[g], width=width - 0.01, color=FIRM_COLORS[g], label=g)
        ax.bar_label(bars, fmt='%.0f', padding=2, fontsize=8, color=INK2)
    ax.set_xticks(x, t.index)
    pct_axis(ax)
    ax.set_ylim(0, t.max().max() * 1.35)
    ax.grid(axis='x', visible=False)
    ax.legend(title='Firm size (workers)', loc='upper left', ncol=4, fontsize=9.5)
    ax.set_title('Pension coverage by race/ethnicity and firm size', pad=24)
    subtitle(ax, 'Share of workers included in a pension plan at work, all years')
    ax.set_xlabel('Race / ethnicity')
    ax.set_ylabel('Share of workers (%)')
    finish(fig, '09_pension_by_race_firm_size')
else:
    print('⚠️  Skipping this graph: no race/ethnicity column in the data')

## 10. Over a career: health insurance by age and firm size
Does the firm-size gap in coverage change as workers get older?

In [ ]:
if 'age_group' in df.columns:
    t = wshare('emp_hins', ['age_group', 'firm']).unstack()[FIRM_ORDER]
    t.index.name, t.columns.name = 'Age group', 'Firm size (workers)'
    show_table(t, 'Employer health insurance (% of workers), by age group and firm size, weighted with asecwt')

    fig, ax = plt.subplots()
    xs = np.arange(len(t))
    for g in FIRM_ORDER:
        ax.plot(xs, t[g], color=FIRM_COLORS[g], label=g, marker='o', markersize=7)
        label_end(ax, xs[-1], t[g].iloc[-1], g)
    ax.set_xticks(xs, t.index.astype(str))
    ax.set_xlim(-0.3, len(t) - 0.4)
    pct_axis(ax)
    ax.set_ylim(0, 100)
    ax.legend(title='Firm size (workers)', loc='upper left', ncol=4, fontsize=9.5)
    ax.set_title('Health insurance by age and firm size', pad=24)
    subtitle(ax, 'Share of workers with employer health insurance, all years')
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('Share of workers (%)')
    finish(fig, '10_insurance_by_age_firm_size')
else:
    print('⚠️  Skipping this graph: no age column in the data')

## Save everything to the Project folder and download the graphs
* Copies every PNG and one zip of all of them to `My Drive/Jobs, Workers, and Development/Project/graphs/`.
* Saves **this notebook** (with its outputs) to the Project folder as `Firmsize_CPS_10_graphs.ipynb`.
* Downloads `firmsize_graphs.zip` to your computer. If it doesn't start, allow downloads for colab.research.google.com.

In [ ]:
import shutil, json
from google.colab import files, _message

os.makedirs(f'{PROJECT}/graphs', exist_ok=True)
for f in saved:
    shutil.copy(f, f'{PROJECT}/graphs/')
zip_path = shutil.make_archive('/content/firmsize_graphs', 'zip', OUT)
shutil.copy(zip_path, f'{PROJECT}/graphs/')
print(f'{len(saved)} graphs saved to {PROJECT}/graphs/:')
for f in saved:
    print('  ', os.path.basename(f))

# Save a copy of this notebook (with outputs) into the Project folder
try:
    nb = _message.blocking_request('get_ipynb', request='', timeout_sec=60)['ipynb']
    with open(f'{PROJECT}/Firmsize_CPS_10_graphs.ipynb', 'w') as fh:
        json.dump(nb, fh)
    print(f'\nNotebook saved to {PROJECT}/Firmsize_CPS_10_graphs.ipynb')
except Exception as e:
    print('\nCould not save the notebook automatically:', e,
          '\nUse File → Save a copy in Drive, then move it to the Project folder.')

files.download(zip_path)